In [91]:
import requests
import pandas as pd
from datetime import datetime

#load teamcolors 

teamcolors = pd.read_csv('nhl_teamcolors.csv')

URL = "https://api-web.nhle.com/v1/standings/now"

# --- Fetch JSON ---
resp = requests.get(URL, timeout=30)
resp.raise_for_status()
data = resp.json()

# standings is a list of team records
standings = data.get("standings", [])

# --- Normalize / flatten ---
df = pd.json_normalize(standings)

# --- Select + rename columns ---
standings_df = (
    df.rename(columns={
        "teamAbbrev.default": "team_id",
        "teamName.default": "team",
        "conferenceName": "conference",
        "divisionName": "division",
        "gamesPlayed": "games_played",
        "wins": "wins",
        "losses": "losses",
        "otLosses": "ot_losses",
        "points": "points",
        "goalDifferential": "goal_diff",
    })[
        [
            "team_id",
            "team",
            "conference",
            "division",
            "games_played",
            "wins",
            "losses",
            "ot_losses",
            "points",
            "goal_diff",
        ]
    ]
    .sort_values(
        by=["points", "wins", "goal_diff"],
        ascending=[False, False, False],
        kind="mergesort"  # stable sort (optional, but nice)
    )
    .reset_index(drop=True)
)

# --- Calculate Olympics standings points --- #
standings_df["olympic_points"] = (
    3 * standings_df["wins"]
    + 2 * (standings_df["losses"])
    + 1 * standings_df["ot_losses"]
)

# --- Sort by Olympics points ---
standings_df = standings_df.sort_values(
    by=["points", "goal_diff"], 
    ascending=[False, False]
).reset_index(drop=True)


# --- Add in pts percentage ---#
standings_df["nhl_pts_perc"] = (
    standings_df["points"] / (2 * standings_df["games_played"])
)

# --- Add rank like row_number() ---
standings_df["rank"] = standings_df.index + 1

# --- Latest standings date (like format(Sys.time(), "%B %d, %Y")) ---
latest_date = datetime.now().strftime("%B %d, %Y")


standings_df.head(15)


,team_id,team,conference,division,games_played,wins,losses,ot_losses,points,goal_diff,olympic_points,nhl_pts_perc,rank
0,COL,Colorado Avalanche,Western,Central,55,37,9,9,83,74,138,0.754545,1
1,TBL,Tampa Bay Lightning,Eastern,Atlantic,55,37,14,4,78,59,143,0.709091,2
2,CAR,Carolina Hurricanes,Eastern,Metropolitan,57,36,15,6,78,34,144,0.684211,3
3,MIN,Minnesota Wild,Western,Central,58,34,14,10,78,27,140,0.672414,4
4,DAL,Dallas Stars,Western,Central,57,34,14,9,77,33,139,0.675439,5
5,MTL,Montréal Canadiens,Eastern,Atlantic,57,32,17,8,72,12,138,0.631579,6
6,DET,Detroit Red Wings,Eastern,Atlantic,58,33,19,6,72,-1,143,0.620690,7
7,PIT,Pittsburgh Penguins,Eastern,Metropolitan,56,29,15,12,70,23,129,0.625000,8
8,BUF,Buffalo Sabres,Eastern,Atlantic,57,32,19,6,70,19,140,0.614035,9
9,BOS,Boston Bruins,Eastern,Atlantic,57,32,20,5,69,13,141,0.605263,10


## Standings Table with Olympic Points

In [92]:
from great_tables import GT, md, google_font


tbl_df = (
    standings_df
      .merge(teamcolors, left_on="team_id", right_on="team_abbr", how="left")
      .loc[:, ["rank", "team_logo_espn", "team", "conference", "division_x",
               "games_played", "wins", "losses", "ot_losses", "points", "goal_diff"]]
)


gt_tbl = (
    GT(tbl_df)
    # render web/local image URLs in cells (ESPN logos)
    .fmt_image(columns="team_logo_espn", height="30px")

    # gradient fill for the Points column
    # (palette is just a list of colors; 2 colors gives a smooth interpolation)
    .data_color(columns="points", palette=["#e3f2fd", "#1565c0"])
    .data_color(columns="olympic_points", palette=["#e3f2fd", "#1565c0"])

    # column labels
    .cols_label(
        rank="Rank",
        team_logo_espn="",
        team="Team",
        conference="Conference",
        division_x="Division",
        games_played="GP",
        wins="W",
        losses="L",
        ot_losses="OTL",
        points="Points",
        goal_diff="Goal Diff",
    )

    # title + subtitle
    .tab_header(
        title=md("**Current NHL Standings**"),
        subtitle=md(f"2025-26 Regular Season. Data as of {latest_date}."),
    )

    # source note
    .tab_source_note(source_note=md("Data: NHL API"))

    # font (Google Fonts helper available)
    .opt_table_font(font=google_font("Outfit"))

    # table-wide theming (your tab_options analog)
    .tab_options(
        column_labels_background_color="white",
        table_border_top_width="3px",
        table_border_top_color="transparent",
        table_border_bottom_width="3px",
        table_border_bottom_color="transparent",
        column_labels_border_top_width="3px",
        column_labels_border_bottom_width="3px",
        column_labels_border_bottom_color="black",
        data_row_padding="3px",
        source_notes_font_size="12px",
        table_font_size="16px",
        heading_align="center",
        heading_background_color="black",
    )
)

#gt_tbl #display table

# Save plot
GT.save(gt_tbl, "NHL Current Standings.png")

GT(_tbl_data=    rank                                     team_logo_espn  \
0      1  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
1      2  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
2      3  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
3      4  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
4      5  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
5      6  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
6      7  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
7      8  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
8      9  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
9     10  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
10    11  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
11    12  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
12    13  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
13    14  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
14    15  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
15    16  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
16    17  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
17    18  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
18    19  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
19    20  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
20    21  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
21    22  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
22    23  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
23    24  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
24    25  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
25    26  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
26    27  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
27    28  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
28    29  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
29    30  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
30    31  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
31    32  https://a.espncdn.com/combiner/i?img=/i/teamlo...   

                     team conference    division_x  games_played  wins  \
0      Colorado Avalanche    Western       Central            55    37   
1     Tampa Bay Lightning    Eastern      Atlantic            55    37   
2     Carolina Hurricanes    Eastern  Metropolitan            57    36   
3          Minnesota Wild    Western       Central            58    34   
4            Dallas Stars    Western       Central            57    34   
5      Montréal Canadiens    Eastern      Atlantic            57    32   
6       Detroit Red Wings    Eastern      Atlantic            58    33   
7     Pittsburgh Penguins    Eastern  Metropolitan            56    29   
8          Buffalo Sabres    Eastern      Atlantic            57    32   
9           Boston Bruins    Eastern      Atlantic            57    32   
10     New York Islanders    Eastern  Metropolitan            58    32   
11   Vegas Golden Knights    Western       Pacific            57    27   
12    Washington Capitals    Eastern  Metropolitan            59    29   
13  Columbus Blue Jackets    Eastern  Metropolitan            56    29   
14           Utah Mammoth    Western       Central            57    30   
15        Edmonton Oilers    Western       Pacific            58    28   
16        Ottawa Senators    Eastern      Atlantic            57    28   
17         Seattle Kraken    Western       Pacific            56    27   
18    Toronto Maple Leafs    Eastern      Atlantic            57    27   
19          Anaheim Ducks    Western       Pacific            56    30   
20    Philadelphia Flyers    Eastern  Metropolitan            56    25   
21       Florida Panthers    Eastern      Atlantic            57    29   
22      Los Angeles Kings    Western       Pacific            56    23   
23    Nashville Predators    Western       Central            57    26   
24        San Jose Sharks    Western       Pacific       

## Individual Team Games

In [93]:
import numpy as np
import pandas as pd
import requests

# get team abbreviations
team_abbrs = (
    teamcolors["team_abbr"]
    .dropna()
    .astype(str)
    .str.lower()
    .unique()
    .tolist()
)

def fetch_team_schedule(team_abbr: str, season: str) -> pd.DataFrame:
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{team_abbr}/{season}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    games_key = next(k for k, v in data.items() if isinstance(v, list))
    df = pd.json_normalize(data[games_key])

    keep = [
        "id", "gameType", "gameDate", "startTimeUTC", "gameState",
        "homeTeam.abbrev", "awayTeam.abbrev",
        "homeTeam.score", "awayTeam.score",
        "gameOutcome.lastPeriodType"
    ]
    keep = [c for c in keep if c in df.columns]
    return df[keep]

season = "20252026"

all_sched = pd.concat([fetch_team_schedule(t, season) for t in team_abbrs], ignore_index=True)

games = all_sched.drop_duplicates(subset=["id"]).copy()

# keep ONLY regular season
games = games[games["gameType"] == 2].copy()

# only completed games
games = games[games["homeTeam.score"].notna() & games["awayTeam.score"].notna()].copy()

games.head()



,id,gameType,gameDate,startTimeUTC,gameState,homeTeam.abbrev,awayTeam.abbrev,homeTeam.score,awayTeam.score,gameOutcome.lastPeriodType
7,2025020021,2,2025-10-09,2025-10-10T02:00:00Z,OFF,SEA,ANA,3.0,1.0,REG
8,2025020036,2,2025-10-11,2025-10-12T02:00:00Z,OFF,SJS,ANA,6.0,7.0,OT
9,2025020056,2,2025-10-14,2025-10-15T02:30:00Z,OFF,ANA,PIT,4.0,3.0,REG
10,2025020070,2,2025-10-16,2025-10-17T02:00:00Z,OFF,ANA,CAR,1.0,4.0,REG
11,2025020091,2,2025-10-19,2025-10-19T23:00:00Z,OFF,CHI,ANA,2.0,1.0,OT


In [97]:
## Turn games into “team-game rows” (2 rows per game) + Olympic points 

games["decided_by"] = games["gameOutcome.lastPeriodType"].fillna("REG").str.upper()

# Home team rows
home_rows = pd.DataFrame({
    "game_id": games["id"],
    "gameDate": games["gameDate"],
    "team_abbr": games["homeTeam.abbrev"].str.lower(),
    "opp_abbr": games["awayTeam.abbrev"].str.lower(),
    "is_home": True,
    "gf": games["homeTeam.score"].astype(int),
    "ga": games["awayTeam.score"].astype(int),
    "decided_by": games["decided_by"],
})

# Away team rows
away_rows = pd.DataFrame({
    "game_id": games["id"],
    "gameDate": games["gameDate"],
    "team_abbr": games["awayTeam.abbrev"].str.lower(),
    "opp_abbr": games["homeTeam.abbrev"].str.lower(),
    "is_home": False,
    "gf": games["awayTeam.score"].astype(int),
    "ga": games["homeTeam.score"].astype(int),
    "decided_by": games["decided_by"],
})

team_games = pd.concat([home_rows, away_rows], ignore_index=True)
team_games["is_win"] = team_games["gf"] > team_games["ga"]

# Olympic 3-2-1 points
team_games["olympic_points"] = np.select(
    [
        team_games["is_win"] & team_games["decided_by"].eq("REG"),
        team_games["is_win"] & team_games["decided_by"].isin(["OT", "SO"]),
        (~team_games["is_win"]) & team_games["decided_by"].isin(["OT", "SO"]),
    ],
    [3, 2, 1],
    default=0
).astype(int)

# add in regulation loss
team_games["reg_loss"] = ((~team_games["is_win"]) & team_games["decided_by"].eq("REG")).astype(int)

# add in OTL
team_games["otl"] = ((~team_games["is_win"]) & team_games["decided_by"].isin(["OT", "SO"])).astype(int)

team_games.head()

,game_id,gameDate,team_abbr,opp_abbr,is_home,gf,ga,decided_by,is_win,olympic_points,reg_loss,otl
0,2025020021,2025-10-09,sea,ana,True,3,1,REG,True,3,0,0
1,2025020036,2025-10-11,sjs,ana,True,6,7,OT,False,1,0,1
2,2025020056,2025-10-14,ana,pit,True,4,3,REG,True,3,0,0
3,2025020070,2025-10-16,ana,car,True,1,4,REG,False,0,1,0
4,2025020091,2025-10-19,chi,ana,True,2,1,OT,True,2,0,0


In [102]:
## Aggregate into an “Olympic standings” table + join teamcolors


# normalize abbreviations so the join works
team_games["team_abbr"] = team_games["team_abbr"].str.upper()
teamcolors["team_abbr"] = teamcolors["team_abbr"].str.upper()

olympic_standings = (
    team_games
    .groupby("team_abbr", as_index=False)
    .agg(
        games_played=("game_id", "count"),
        wins=("is_win", "sum"),
        losses=("reg_loss", "sum"),      # <-- regulation losses (this matches NHL "L")
        ot_losses=("otl", "sum"),        # <-- matches NHL "OTL"
        gf=("gf", "sum"),
        ga=("ga", "sum"),
        olympic_points=("olympic_points", "sum"),
    )
)

# calculate GD
olympic_standings["goal_diff"] = olympic_standings["gf"] - olympic_standings["ga"]

# calculate pts % for Olympic system
olympic_standings["olympic_pts_perc"] = (
    olympic_standings["olympic_points"] / (3 * olympic_standings["games_played"])
)

# join team metadata (logos, names, colors, division, etc.)
olympic_standings = olympic_standings.merge(
    teamcolors,
    on="team_abbr",
    how="left"
)

# rank + sort (customize tie-breakers however you want)
olympic_standings = (
    olympic_standings
    .sort_values(["wins", "goal_diff"], ascending=False)
    .reset_index(drop=True)
)
olympic_standings["rank"] = olympic_standings.index + 1

#create lookup to add in Conference
team_lookup = (
    standings_df[["team_id", "conference", "points", "nhl_pts_perc"]]
    .rename(columns={"team_id": "team_abbr"})
    .copy()
)

# normalize keys
team_lookup["team_abbr"] = team_lookup["team_abbr"].str.upper()

# join in Conference from standings_df 
olympic_standings = olympic_standings.merge(team_lookup, on="team_abbr", how="left")

# sort again
olympic_standings = (
    olympic_standings
    .sort_values(["olympic_points", "olympic_pts_perc"], ascending=False)
    .reset_index(drop=True)
)

olympic_standings["olympic_rank"] = olympic_standings.index + 1

olympic_standings

,team_abbr,games_played,wins,losses,ot_losses,gf,ga,olympic_points,goal_diff,olympic_pts_perc,...,location,mascot,sportslogos_name,team_logo_espn,logo,rank,conference,points,nhl_pts_perc,olympic_rank
0,COL,55,37,9,9,212,138,117,74,0.709091,...,Colorado,Avalanche,Colorado Avalanche,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/8/thumb...,1,Western,83,0.754545,1
1,TBL,55,37,14,4,199,140,106,59,0.642424,...,Tampa Bay,Lightning,Tampa Bay Lightning,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/27/thum...,2,Eastern,78,0.709091,2
2,DAL,57,34,14,9,193,160,105,33,0.614035,...,Dallas,Stars,Dallas Stars,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/10/thum...,4,Western,77,0.675439,3
3,CAR,57,36,15,6,197,163,104,34,0.608187,...,Carolina,Hurricanes,Carolina Hurricanes,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/6/thumb...,3,Eastern,78,0.684211,4
4,MIN,58,34,14,10,196,169,99,27,0.568966,...,Minnesota,Wild,Minnesota Wild,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/15/thum...,5,Western,78,0.672414,5
5,BUF,57,32,19,6,195,176,96,19,0.561404,...,Buffalo,Sabres,Buffalo Sabres,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/4/thumb...,7,Eastern,70,0.614035,6
6,PIT,56,29,15,12,192,169,95,23,0.565476,...,Pittsburgh,Penguins,Pittsburgh Penguins,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/24/thum...,13,Eastern,70,0.625000,7
7,DET,58,33,19,6,174,175,95,-1,0.545977,...,Detroit,Red Wings,Detroit Red Wings,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/11/thum...,6,Eastern,72,0.620690,8
8,BOS,57,32,20,5,195,182,93,13,0.543860,...,Boston,Bruins,Boston Bruins,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/3/thumb...,8,Eastern,69,0.605263,9
9,MTL,57,32,17,8,199,187,93,12,0.543860,...,Montreal,Canadiens,Montreal Canadiens,https://a.espncdn.com/combiner/i?img=/i/teamlo...,http://content.sportslogos.net/logos/1/16/thum...,9,Eastern,72,0.631579,10


In [ ]:
## build GT table

tbl_df = (
    olympic_standings
      .loc[:, ["olympic_rank", "rank", "team_logo_espn", "name", "division", "conference",
               "games_played", "wins", "losses", "ot_losses", "goal_diff", "points", "nhl_pts_perc", 
               "olympic_points", "olympic_pts_perc"]]
)

# round pts % columns first
tbl_df["nhl_pts_perc"] = tbl_df["nhl_pts_perc"].round(3)
tbl_df["olympic_pts_perc"] = tbl_df["olympic_pts_perc"].round(3)

# remove leading zero (0.755 -> .755)
tbl_df["nhl_pts_perc"] = tbl_df["nhl_pts_perc"].map(lambda x: f"{x:.3f}".lstrip("0"))
tbl_df["olympic_pts_perc"] = tbl_df["olympic_pts_perc"].map(lambda x: f"{x:.3f}".lstrip("0"))


gt_tbl = (
    GT(tbl_df)
    # render web/local image URLs in cells (ESPN logos)
    .fmt_image(columns="team_logo_espn", height="30px")

    # gradient fill for the Points column
    # (palette is just a list of colors; 2 colors gives a smooth interpolation)
    .data_color(columns="olympic_points", palette=["#e3f2fd", "#1565c0"])
    .data_color(columns="points", palette=["#e3f2fd", "#1565c0"])

    # column labels
    .cols_label(
        olympic_rank="olympic_rank",
        rank="NHL Rank",
        team_logo_espn="",
        name="Team",
        conference="Conference",
        division="Division",
        games_played="GP",
        wins="W",
        losses="L",
        ot_losses="OTL",
        points="NHL Pts",
        nhl_pts_perc="NHL Pts%",
        olympic_points="OLY Pts",
        olympic_pts_perc="OLY Pts%",
        goal_diff="Goal Diff",
    )

    # title + subtitle
    .tab_header(
        title=md("**NHL Standings using Olympic Points**"),
        subtitle=md(f"2025-26 Regular Season. Data as of {latest_date}."),
    )

    # source note
    .tab_source_note(source_note=md("Data: NHL API"))

    # font (Google Fonts helper available)
    .opt_table_font(font=google_font("Outfit"))

    # table-wide theming (your tab_options analog)
    .tab_options(
        column_labels_background_color="white",
        table_border_top_width="3px",
        table_border_top_color="transparent",
        table_border_bottom_width="3px",
        table_border_bottom_color="transparent",
        column_labels_border_top_width="3px",
        column_labels_border_bottom_width="3px",
        column_labels_border_bottom_color="black",
        data_row_padding="3px",
        source_notes_font_size="12px",
        table_font_size="16px",
        heading_align="center",
        heading_background_color="black",
    )
)

gt_tbl #display plot in viewer

# Save plot
GT.save(gt_tbl, "NHL Olympics Points.png", scale=3)

GT(_tbl_data=    rank                                     team_logo_espn  \
0      1  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
1      2  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
2      3  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
3      4  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
4      5  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
5      6  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
6      7  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
7      8  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
8      9  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
9     10  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
10    11  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
11    12  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
12    13  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
13    14  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
14    15  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
15    16  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
16    17  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
17    18  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
18    19  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
19    20  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
20    21  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
21    22  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
22    23  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
23    24  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
24    25  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
25    26  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
26    27  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
27    28  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
28    29  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
29    30  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
30    31  https://a.espncdn.com/combiner/i?img=/i/teamlo...   
31    32  https://a.espncdn.com/combiner/i?img=/i/teamlo...   

                     name      division conference  games_played  wins  \
0      Colorado Avalanche       Central    Western            55    37   
1     Tampa Bay Lightning      Atlantic    Eastern            55    37   
2     Carolina Hurricanes  Metropolitan    Eastern            57    36   
3            Dallas Stars       Central    Western            57    34   
4          Minnesota Wild       Central    Western            58    34   
5       Detroit Red Wings      Atlantic    Eastern            58    33   
6          Buffalo Sabres      Atlantic    Eastern            57    32   
7           Boston Bruins      Atlantic    Eastern            57    32   
8      Montreal Canadiens      Atlantic    Eastern            57    32   
9      New York Islanders  Metropolitan    Eastern            58    32   
10           Utah Mammoth       Pacific    Western            57    30   
11          Anaheim Ducks       Pacific    Western            56    30   
12    Pittsburgh Penguins  Metropolitan    Eastern            56    29   
13    Washington Capitals  Metropolitan    Eastern            59    29   
14  Columbus Blue Jackets  Metropolitan    Eastern            56    29   
15       Florida Panthers      Atlantic    Eastern            57    29   
16        Ottawa Senators      Atlantic    Eastern            57    28   
17        Edmonton Oilers       Pacific    Western            58    28   
18      New Jersey Devils  Metropolitan    Eastern            57    28   
19   Vegas Golden Knights       Pacific    Western            57    27   
20         Seattle Kraken       Pacific    Western            56    27   
21    Toronto Maple Leafs      Atlantic    Eastern            57    27   
22        San Jose Sharks       Pacific    Western            55    27   
23    Nashville Predators       Central    Western            57    26   
24    Philadelphia Flyers  Metropolitan    Eastern       

In [100]:
# write Olympic Standings datasets to csv

export_df = (
    tbl_df
    .sort_values("rank")
    .reset_index(drop=True)
)

export_df.to_csv("olympic_points_standings.csv", index=False)

export_df.head(10)

,rank,team_logo_espn,name,division,conference,games_played,wins,losses,ot_losses,goal_diff,points,nhl_pts_perc,olympic_points,olympic_pts_perc
0,1,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Colorado Avalanche,Central,Western,55,37,9,9,74,83,.755,117,.709
1,2,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Tampa Bay Lightning,Atlantic,Eastern,55,37,14,4,59,78,.709,106,.642
2,3,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Carolina Hurricanes,Metropolitan,Eastern,57,36,15,6,34,78,.684,104,.608
3,4,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Dallas Stars,Central,Western,57,34,14,9,33,77,.675,105,.614
4,5,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Minnesota Wild,Central,Western,58,34,14,10,27,78,.672,99,.569
5,6,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Detroit Red Wings,Atlantic,Eastern,58,33,19,6,-1,72,.621,95,.546
6,7,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Buffalo Sabres,Atlantic,Eastern,57,32,19,6,19,70,.614,96,.561
7,8,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Boston Bruins,Atlantic,Eastern,57,32,20,5,13,69,.605,93,.544
8,9,https://a.espncdn.com/combiner/i?img=/i/teamlo...,Montreal Canadiens,Atlantic,Eastern,57,32,17,8,12,72,.632,93,.544
9,10,https://a.espncdn.com/combiner/i?img=/i/teamlo...,New York Islanders,Metropolitan,Eastern,58,32,21,5,7,69,.595,91,.523
